# Kapitel 19.6 - Mini-Projekt: Kursportal mit CGI- und WSGI-Denke

In diesem Notebook entwickelst du ein kleines, aber realitaetsnahes Kursportal.
Der Fokus liegt auf sauberer Struktur, valider Eingabeverarbeitung und nachvollziehbarer Architektur.

# Lernziele

- ein Mini-Projekt in klaren Modulen aufbauen
- Formular- und Query-Daten robust verarbeiten
- HTML- und JSON-Antworten situationsgerecht einsetzen
- typische Projektentscheidungen begruenden

# Voraussetzungen

- Kapitel 19.1 bis 19.5
- sichere Grundlagen in Funktionen, Dictionaries und Fehlerbehandlung

# Theorie

Ein Mini-Projekt sollte bereits dieselben Qualitaetsmerkmale wie ein groesseres System haben:

1. klare Verantwortlichkeiten
2. wiederverwendbare Hilfsfunktionen
3. zentrale Validierung
4. konsistente Responses

Gerade in Lernprojekten entsteht dadurch ein professioneller Denkstil.

# Erklaerung

Wir modellieren ein Kursportal mit drei Hauptfaellen:

- Kursliste anzeigen
- Einzelkurs anzeigen
- Anmeldung verarbeiten

Technisch bleibt alles absichtlich framework-neutral, damit die Kernlogik transparent bleibt.

# Syntax

```python
def portal_app(environ, start_response):
    ...
    return [body_bytes]
```

# Merke

- Ein Projekt wird nicht durch Dateianzahl professionell, sondern durch Strukturqualitaet.
- Jede Eingabe aus Requests gilt als untrusted input.

# Parameter

Wichtige Projektparameter in diesem Notebook:

- `PATH_INFO` fuer Routing
- `QUERY_STRING` fuer Filter und IDs
- Formularfelder `name`, `email`, `kurs_id`

# Rueckgabewert

Die Haupt-App liefert ein WSGI-konformes bytes-Iterable zurueck.
Interne Hilfsfunktionen liefern Strings, Dictionaries oder Tupel fuer bessere Testbarkeit.

In [ ]:
# Beispiel 1: Projekt-Datenmodell
KURSE = [
    {'id': 1, 'titel': 'Python Grundlagen', 'dauer_wochen': 6},
    {'id': 2, 'titel': 'Web mit WSGI', 'dauer_wochen': 4},
    {'id': 3, 'titel': 'Datenanalyse Einstieg', 'dauer_wochen': 8}
]

for kurs in KURSE:
    print(f"[{kurs['id']}] {kurs['titel']} ({kurs['dauer_wochen']} Wochen)")

# Beispiel 1 - Erklaerung

Ein bewusst einfaches Datenmodell reicht fuer viele didaktische Ziele aus.
Wichtig ist die einheitliche Struktur, damit spaetere Funktionen klar darauf zugreifen koennen.

In [ ]:
# Beispiel 2: Hilfsfunktionen fuer Suche und Validierung
def finde_kurs_nach_id(kurs_id):
    for kurs in KURSE:
        if kurs['id'] == kurs_id:
            return kurs
    return None

def validiere_anmeldung(name, email, kurs_id_text):
    fehler = []

    if not name.strip():
        fehler.append('Name ist erforderlich.')

    if '@' not in email or '.' not in email:
        fehler.append('E-Mail ist ungueltig.')

    if not kurs_id_text.isdigit():
        fehler.append('kurs_id muss numerisch sein.')
    else:
        kurs = finde_kurs_nach_id(int(kurs_id_text))
        if kurs is None:
            fehler.append('Kurs wurde nicht gefunden.')

    return fehler

print(validiere_anmeldung('Elif', 'elif@mail.de', '2'))
print(validiere_anmeldung('', 'falsch', '99'))

# Beispiel 2 - Erklaerung

Die Trennung zwischen Kurssuche und Formularvalidierung macht den Code wartbar.
Beide Funktionen lassen sich isoliert testen.

In [ ]:
# Beispiel 3: WSGI-App fuer das Kursportal (simuliert)
from urllib.parse import parse_qs
import json

def json_response(start_response, data, status='200 OK'):
    body = json.dumps(data, ensure_ascii=False).encode('utf-8')
    start_response(status, [('Content-Type', 'application/json; charset=utf-8')])
    return [body]

def portal_app(environ, start_response):
    path = environ.get('PATH_INFO', '/')

    if path == '/api/kurse':
        return json_response(start_response, {'kurse': KURSE})

    if path == '/api/kurs':
        query = parse_qs(environ.get('QUERY_STRING', ''))
        kurs_id_text = query.get('id', [''])[0]

        if not kurs_id_text.isdigit():
            return json_response(start_response, {'fehler': 'id muss numerisch sein'}, '400 Bad Request')

        kurs = finde_kurs_nach_id(int(kurs_id_text))
        if kurs is None:
            return json_response(start_response, {'fehler': 'Kurs nicht gefunden'}, '404 Not Found')

        return json_response(start_response, {'kurs': kurs})

    return json_response(start_response, {'fehler': 'Route unbekannt'}, '404 Not Found')

def fake_start_response(status, headers):
    print('STATUS:', status)

print(portal_app({'PATH_INFO': '/api/kurse'}, fake_start_response)[0].decode('utf-8'))
print(portal_app({'PATH_INFO': '/api/kurs', 'QUERY_STRING': 'id=2'}, fake_start_response)[0].decode('utf-8'))

# Praxisbeispiel

Erweitere das Portal um Anmeldung:

- Route `/api/anmeldung`
- Pflichtfelder validieren
- Erfolgs- oder Fehlerobjekt als JSON liefern

Diese Aufgabe simuliert bereits einen typischen Produktiv-Workflow.

In [ ]:
# Praxis-Loesung
def verarbeite_anmeldung(query_string):
    daten = parse_qs(query_string, keep_blank_values=True)
    name = daten.get('name', [''])[0].strip()
    email = daten.get('email', [''])[0].strip()
    kurs_id_text = daten.get('kurs_id', [''])[0].strip()

    fehler = validiere_anmeldung(name, email, kurs_id_text)
    if fehler:
        return {'ok': False, 'fehler': fehler}

    kurs = finde_kurs_nach_id(int(kurs_id_text))
    return {
        'ok': True,
        'nachricht': 'Anmeldung gespeichert',
        'teilnehmer': name,
        'kurs': kurs['titel']
    }

print(verarbeite_anmeldung('name=Deniz&email=deniz%40mail.de&kurs_id=1'))
print(verarbeite_anmeldung('name=&email=kaputt&kurs_id=99'))

# Haeufige Fehler

1. Route-Logik und Fachlogik in einer riesigen Funktion mischen.
2. Keine einheitliche Fehlerstruktur definieren.
3. IDs als Text weiterreichen, ohne Typumwandlung.
4. Antwortformate je Endpoint unterschiedlich machen.

# Best Practice

- Lege zuerst Response-Format und Fehlerobjekte fest.
- Schreibe kleine Testfaelle fuer Validierung und Suche.
- Nutze konsistente Benennung in Datenobjekten.
- Trenne Unterrichtscode (didaktisch klar) von Produktionsdetails (z. B. echte Persistenz).

# Tipp

Wenn dein Code waechst, fuehre ein kleines Projektlayout ein:
`routing.py`, `services.py`, `validation.py`, `responses.py`.
Das macht den Wechsel zu Flask oder Django spaeter deutlich leichter.

# Uebung

Erweitere das Projekt um eine Suchfunktion `/api/suche?text=...`, die nur passende Kurse liefert.
Achte auf:
- leeren Suchtext
- Gross/Kleinschreibung
- einheitliches JSON-Ergebnis

In [ ]:
# Loesung zur Uebung
def suche_kurse(text):
    suchtext = text.strip().lower()
    if not suchtext:
        return []
    return [kurs for kurs in KURSE if suchtext in kurs['titel'].lower()]

print(suche_kurse('python'))
print(suche_kurse('web'))
print(suche_kurse('   '))

# Zusammenfassung

Du hast ein vollstaendiges Lernprojekt erstellt, das bereits professionelle Prinzipien nutzt:

- sauberes Routing
- zentrale Validierung
- konsistente JSON-Responses
- klare Erweiterungspunkte

# Weiterfuehrende Links

- PEP 3333 (WSGI)
- Werkzeug Routing Konzepte
- Flask Blueprint Architektur

## Technischer Tiefgang

Der Fokus liegt auf reproduzierbaren technischen Entscheidungen statt auf isolierten Einzelbeispielen.
Dabei werden Architektur, Robustheit und Betriebsfaehigkeit gemeinsam betrachtet.

## Zentrale Fachbegriffe

HTTP Semantics
Status Code Family
WSGI Callable
Request Lifecycle
Input Sanitization
Header Validation

In [ ]:
# WSGI-Minibeispiel mit Statuscode
def app(environ, start_response):
    path = environ.get("PATH_INFO", "/")
    if path == "/health":
        start_response("200 OK", [("Content-Type", "text/plain")])
        return [b"ok"]
    start_response("404 Not Found", [("Content-Type", "text/plain")])
    return [b"not found"]

## Fallstudie (Praxis)

Waehle ein realistisches Produktionsszenario und beschreibe systematisch Ursache, Risiko und technische Gegenmassnahmen.
Ergaenze mindestens ein Kriterium fuer Monitoring und ein Kriterium fuer Release-Entscheidungen.

## Haeufige Fehler und Debugging-Checkliste

- Ist das Problem reproduzierbar mit klaren Schritten?
- Sind relevante Signale vorhanden (Logs, Tests, Metriken)?
- Wurde eine konkrete Hypothese getestet und falsifiziert/bestaetigt?
- Ist die Korrektur durch einen Regressionstest abgesichert?
- Wurden Betriebsfolgen und Dokumentation mit aktualisiert?

## Pruefungsfragen und Kurzloesungen

1. Warum ist Reproduzierbarkeit in Fehleranalyse und Betrieb zentral?
Kurzloesung: Ohne reproduzierbare Befunde sind Ursachenanalyse, Fix und Absicherung nicht belastbar.
2. Was unterscheidet technische Begriffe von bloessem Buzzword-Einsatz?
Kurzloesung: Praezise Begriffe steuern messbare Entscheidungen und verbessern Teamkommunikation.
3. Welche Mindestkriterien sollte ein Release-Gate enthalten?
Kurzloesung: Teststatus, Sicherheitschecks, Fehlerbudget und nachvollziehbare Freigabeentscheidung.